# AI-Based Heart Anomaly Detection Using ECG Signals
## 01. Exploratory Data Analysis & Baseline Inspection

**Dataset:** MIT-BIH Arrhythmia Database (`mitdb` v1.0.0, PhysioNet)  
**Standard:** AAMI EC57 5-Class Heartbeat Super-Classes (N, S, V, F, Q)  
**Inter-Patient Protocol:** De Chazal et al. (2004) DS1 (Train) and DS2 (Test) separation

---
### Objectives of this Notebook
1. Verify dependencies and reproducibility parameters.
2. Download and load official MIT-BIH recordings using `wfdb`.
3. Inspect signal headers, leads (MLII vs V1/V5), and sampling frequency (360 Hz).
4. Analyze reference annotation symbols and map them to AAMI EC57 classes.
5. Examine signal filtering (zero-phase Butterworth 0.5–40 Hz) and baseline wander removal.
6. Segment heartbeats around R-peaks and visualize morphology per class.

In [ ]:
import sys
from pathlib import Path

# Ensure project root is in sys.path
project_root = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import wfdb
import torch

from src.data_loader import download_records, load_record, inspect_record, get_default_data_dir
from src.preprocessing import preprocess_record, AAMI_CLASSES, AAMI_MAPPING
from src.visualization import plot_signal_filtering_comparison, plot_heartbeat_morphologies, plot_class_distribution

print(f"Python: {sys.version.split()[0]}")
print(f"WFDB:   {wfdb.__version__}")
print(f"Torch:  {torch.__version__}")

### 1. Download Sample Records
We start by downloading Record `100` and Record `101` from the official PhysioNet repository into `data/raw/`.

In [ ]:
data_dir = get_default_data_dir()
downloaded = download_records(["100", "101"], output_dir=data_dir)
print(f"Available records in {data_dir}: {downloaded}")

### 2. Inspect Record Metadata and Annotation Symbols
Let us examine Record `100` header properties and the exact symbols present in the `.atr` file.

In [ ]:
record, annotation = load_record("100", data_dir=data_dir)
info = inspect_record(record, annotation)

print(f"Record Name:     {info['record_name']}")
print(f"Sampling Rate:   {info['sampling_rate_hz']} Hz")
print(f"Duration:        {info['duration_minutes']:.2f} minutes ({info['total_samples']} samples)")
print(f"Available Leads: {info['leads']}")
print(f"Total Beats:     {info['total_annotations']}")
print(f"Raw Symbol Counts: {info['symbol_counts']}")

### 3. AAMI EC57 Class Mapping
PhysioNet MIT-BIH annotation symbols and their standard mapping into the 5 AAMI heartbeat classes:

| AAMI Class | Description | MIT-BIH Symbols |
|---|---|---|
| **N** | Normal / Non-ectopic | `N`, `L`, `R`, `e`, `j` |
| **S** | Supraventricular ectopic (SVEB) | `A`, `a`, `J`, `S` |
| **V** | Ventricular ectopic (VEB) | `V`, `E` |
| **F** | Fusion of ventricular and normal | `F` |
| **Q** | Paced / Unclassifiable | `/`, `f`, `Q`, `?` |

In [ ]:
# Preprocess record 100: bandpass filter (0.5-40 Hz) and Z-score normalize
rec100 = preprocess_record("100", data_dir=data_dir, preferred_lead="MLII")
print(f"Segmented {rec100['num_beats']} heartbeats from Record 100 (Lead: {rec100['lead_name']})")
print(f"Segment matrix shape: {rec100['segments'].shape}")

### 4. Waveform & Preprocessing Inspection
Comparison of Raw vs Bandpass Filtered & Normalized ECG signals with annotated R-peaks.

In [ ]:
fig_filter = plot_signal_filtering_comparison(rec100, start_sec=10.0, duration_sec=5.0, show=True)

### 5. Heartbeat Morphology & Class Distribution
Examine average waveform signatures and class imbalance.

In [ ]:
fig_morph = plot_heartbeat_morphologies(rec100["segments"], rec100["labels"], show=True)
fig_dist = plot_class_distribution(rec100["labels"], title_suffix="(Record 100)", show=True)